# Superposition

# Superposition

## What This Is
Polysemanticity: individual neurons respond to multiple unrelated concepts. This occurs because neural networks need to represent more features than they have dimensions. The solution: *superposition* — multiple features are packed into the same neuron.

## Why Superposition Occurs
Sparsity is the key. If features are rarely active simultaneously, they can share neurons without interference. A neuron can respond to feature A in context C1 and feature B in context C2 if A and B rarely co-occur.

## The Toy Model of Superposition
Eltman & Olsson (2022) demonstrate this with a minimal setup: a 2-layer MLP where a bottleneck dimension forces 5 features into 2 dimensions. When features are sparse, the model learns to pack all 5 features in a pentagon arrangement.

In [ ]:
import numpy as np

np.random.seed(42)

# Toy model of superposition
# 5 features -> 2D bottleneck -> reconstruct 5 features
# When features are sparse, the model packs them in superposition

N_FEATURES = 5
N_HIDDEN = 2  # bottleneck

def generate_sparse_features(n: int, n_features: int, sparsity: float = 0.9) -> np.ndarray:
    """Generate sparse feature vectors (most features are 0)."""
    X = np.random.uniform(0, 1, (n, n_features))
    mask = np.random.random((n, n_features)) < sparsity  # 90% sparse
    return X * (1 - mask.astype(float))

def relu_ae_forward(X, W, b_hid, b_out):
    h = np.maximum(0, X @ W + b_hid)
    return h @ W.T + b_out  # weight-tied decoder

# Train a toy autoencoder with bottleneck
X_train = generate_sparse_features(10000, N_FEATURES, sparsity=0.9)
X_test = generate_sparse_features(500, N_FEATURES, sparsity=0.9)

# Arrange W as pentagon (what theory predicts for superposition)
# Each column is a direction for one feature
angles = np.linspace(0, 2*np.pi, N_FEATURES, endpoint=False)
W_theory = np.column_stack([np.cos(angles), np.sin(angles)])  # 2 x 5

# Train
W = np.random.randn(N_FEATURES, N_HIDDEN) * 0.3
b_hid = np.zeros(N_HIDDEN)
b_out = np.zeros(N_FEATURES)
lr = 0.01

for epoch in range(500):
    idx = np.random.randint(0, len(X_train), 128)
    batch = X_train[idx]
    h = np.maximum(0, batch @ W + b_hid)
    recon = h @ W.T + b_out
    err = recon - batch
    # Gradient
    dout = 2 * err / len(batch)
    db_out = dout.mean(0)
    dh = dout @ W  * (h > 0)
    db_hid = dh.mean(0)
    dW = batch.T @ dh / len(batch) + h.T @ dout / len(batch)
    W -= lr * dW; b_hid -= lr * db_hid; b_out -= lr * db_out

# Visualize learned directions
W_norm = W / (np.linalg.norm(W, axis=1, keepdims=True) + 1e-8)
print('Superposition Demo: Feature Directions in 2D')
print(f'5 features packed into 2D bottleneck')
print(f'\nLearned feature directions (normalized):')
for i in range(N_FEATURES):
    print(f'  Feature {i}: ({W_norm[i, 0]:+.3f}, {W_norm[i, 1]:+.3f})')

# Measure inter-feature cosine similarity
gram = W_norm @ W_norm.T
off_diag = gram[~np.eye(N_FEATURES, dtype=bool)]
print(f'\nAvg |cosine similarity| between features: {np.abs(off_diag).mean():.3f}')
print('(In perfect superposition, all pairs are ~cos(72°)=0.31 for pentagon)')
